# Лабораторная работа №3

## Тема
Метод ближайших соседей, подбор гиперпараметров и кросс-валидация.

## Цель работы
Построить и улучшить модель KNN для задачи классификации с помощью GridSearchCV и RandomizedSearchCV.

## Краткая теория

KNN относит объект к классу большинства среди `K` ближайших соседей в пространстве признаков.
Качество модели чувствительно к масштабу признаков и значению `K`, поэтому используют стандартизацию и подбор гиперпараметров с кросс-валидацией.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
DATA_PATH = "../data/breast_cancer_wisconsin/data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [2]:
# Подготовка данных
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")
df["diagnosis"] = df["diagnosis"].replace({"M": 1, "B": 0})
df = df[df["diagnosis"].isin([0, 1])].copy()
df["diagnosis"] = df["diagnosis"].astype(int)

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

((455, 30), (114, 30))

In [3]:
# Базовая модель KNN с произвольным K
base_k = 7
base_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=base_k))
])
base_model.fit(X_train, y_train)
base_pred = base_model.predict(X_test)

base_metrics = {
    "accuracy": accuracy_score(y_test, base_pred),
    "f1": f1_score(y_test, base_pred)
}

print("Base K:", base_k)
print(base_metrics)
print("\nClassification report:\n", classification_report(y_test, base_pred))

Base K: 7
{'accuracy': 0.956140350877193, 'f1': 0.9382716049382716}

Classification report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97        72
           1       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [4]:
# Подбор K через GridSearchCV и RandomizedSearchCV
search_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

param_grid = {
    "model__n_neighbors": list(range(1, 41)),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv2 = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=search_pipeline,
    param_grid=param_grid,
    cv=cv1,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=cv2,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print("Grid best params:", grid_search.best_params_)
print("Grid CV f1:", grid_search.best_score_)
print("Random best params:", random_search.best_params_)
print("Random CV f1:", random_search.best_score_)

Grid best params: {'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'uniform'}
Grid CV f1: 0.966707720140556
Random best params: {'model__weights': 'distance', 'model__p': 2, 'model__n_neighbors': 4}
Random CV f1: 0.9565131058062747


In [5]:
# Оценка оптимальной модели на тестовой выборке
best_model = grid_search.best_estimator_ if grid_search.best_score_ >= random_search.best_score_ else random_search.best_estimator_
opt_pred = best_model.predict(X_test)

opt_metrics = {
    "accuracy": accuracy_score(y_test, opt_pred),
    "f1": f1_score(y_test, opt_pred)
}

comparison = pd.DataFrame([base_metrics, opt_metrics], index=["Base KNN", "Optimized KNN"])
comparison

,accuracy,f1
Base KNN,0.956140,0.938272
Optimized KNN,0.964912,0.950000


## Выводы

1. Построена базовая модель KNN и оценена по метрикам `accuracy` и `f1`.
2. Подбор гиперпараметров выполнен двумя методами: `GridSearchCV` и `RandomizedSearchCV`.
3. Применены две стратегии кросс-валидации: `StratifiedKFold` и `RepeatedStratifiedKFold`.
4. Выполнено сравнение качества исходной и оптимальной моделей.